# Cole-Parmer GenoGrinder (fka SPEX GenoGrinder)

The GenoGrinder is a high-throughput plate shaker and homogenizer. It clamps a microplate (or a stack of plates, or a set of vials in a plate-format holder) between two platens and shakes it vertically for a fixed time at a fixed speed, grinding or mixing the contents of every well at once.

Product page: [Cole-Parmer SamplePrep HG-600-230 Geno/Grinder 2010](https://www.coleparmer.com/i/cole-parmer-sampleprep-hg-600-230-geno-grinder-2010-tissue-homogenizer-and-cell-lyser-230-vac-50-hz/0457684)

| Property | Value |
|---|---|
| Communication | Serial, ASCII line protocol |
| Serial settings | 9600 baud, 8 data bits, no parity, 1 stop bit |
| Line terminator | carriage return (`\r`) |
| Duration range | 1--999 s |
| Speed range | 1--9999 rpm |
| Clamp | motorized (open/close) or fixed, depending on model |

```{warning}
This driver has NOT been tested against hardware in PyLabRobot. `setup()` logs a
warning to that effect. If you verify it on your machine, please open a PR to
remove the warning.
```

## Physical setup

Connect the GenoGrinder's serial port to your computer, typically through a USB-to-serial adapter, and note the port name (`/dev/ttyUSB*` on Linux, `/dev/tty.usbserial-*` on macOS, `COM*` on Windows).

Make sure the instrument's serial parameters match the driver defaults (9600 baud, 8 data bits, no parity, 1 stop bit), and that the safety lid is closed -- the machine will not run with it open.

## Connect

`setup()` opens the serial port and runs the instrument's power-on routine, which homes the mechanism and readies the clamp. Its reply doubles as a communication check, so a successful `setup()` means the machine is talking and ready to run.

Some GenoGrinder models have a fixed clamp that you tighten by hand instead of a motorized one. On those, pass `use_clamp_commands=False`: `open_clamp()` and `close_clamp()` then do nothing, so the same protocol code runs on either machine.

In [ ]:
from pylabrobot.cole_parmer import GenoGrinder

grinder = GenoGrinder(port="/dev/ttyUSB0")  # replace with your port
await grinder.setup()

## Check the clamp

`request_clamp_state()` reports whether the clamp is `"open"`, `"closed"`, or `"unknown"`.

In [ ]:
print("Clamp:", await grinder.request_clamp_state())

## Open the clamp

`open_clamp()` reads the current position and only moves if the clamp is not already open, so it is safe to call repeatedly.

In [ ]:
await grinder.open_clamp()

## Load a plate

Place your plate (or plate stack) in the clamp now, centered on the platen, and close the safety lid.

## Close the clamp

`close_clamp()` clamps the plate down. Like `open_clamp()`, it checks the current position first.

In [ ]:
await grinder.close_clamp()

## Shake

`shake()` uploads the run parameters, starts the run, then polls the instrument until it reports the run is complete -- so the call returns when the plate has actually finished shaking.

`duration` is in seconds (1--999) and `speed` in rpm (1--9999). Start gentle and work up: hard grinding at high rpm can crack plates and unseat seals.

In [ ]:
await grinder.shake(duration=30, speed=1500)

## Read the status

`request_status()` returns the instrument's raw status line -- `Standby` when idle, or one of the transient run states (`Running Sample`, `Locking`, `Mixing`, `Unlocking`, `Run Complete`) during a run. Useful when driving the machine from another task while a run is going.

In [ ]:
print("Status:", await grinder.request_status())

## Abort a run

`stop_shaking()` stops a run in progress. Because `shake()` blocks until the run finishes, call this from a separate task (or after interrupting the cell) when you need to cut a run short.

In [ ]:
await grinder.stop_shaking()

## Home the clamp

`home_clamp()` returns the clamp to its reference position. Use it to recover a known state after an aborted run.

In [ ]:
await grinder.home_clamp()

## Unload the plate

Open the clamp again and take the plate out.

In [ ]:
await grinder.open_clamp()

## Teardown

`stop()` halts any run still in progress and closes the serial connection.

In [ ]:
await grinder.stop()